# Resultado - Mortalidade por faixa etária

In [ ]:
import re
import unicodedata

import geobr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import plotly.io as pio
pio.renderers.default = "notebook"

import numpy as np
from IPython.display import Markdown, display


In [ ]:
CSV_MORTALIDADE = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"

MUNICIPIO = None
CODIGO_MUNICIPIO = None
REGIONAL_SAUDE = None

ANOS_RANGE_MORTALIDADE = (1979, 2024)
ANOS_PASSO_MORTALIDADE = 1

# Acumulado em blocos (para reproduzir 1979..1983 => rótulo 1983)
ANOS_POR_BLOCO_ACUMULADO = 5

# Mapa de MG (média anual por município)
DIVISOR_MEDIA_ANUAL_MUNICIPIO = 45 # quantidade de anos
ANO_MALHA_MUNICIPIOS_MG = 2020
ESCALA_CORES_MAPA = "YlOrRd"

FAIXAS_ETARIAS_MORTALIDADE = [
    "0-9", "10-19", "20-29", "30-39",
    "40-49", "50-59", "60-69", "70 ou mais",
]

COR_BARRAS = "#6ca8e5"
COR_PONTOS = "#d73f3f"
LARGURA_GRAFICO = 1900
ALTURA_GRAFICO_MORTALIDADE = 1300  
X_MIN = 0
X_MAX = 13000

FONTE_TITULO = 22
FONTE_EIXO_TITULO = 16
FONTE_EIXO_TICK = 14
FONTE_LEGENDA = 14
FONTE_TEXTO_BARRA = 14
FONTE_TEXTO_PONTO = 12

TITULO_MORTALIDADE = "Mortalidade por faixa etária"
TITULO_MORTALIDADE_ACUM = "Mortalidade por faixa etária"

In [ ]:
# 3) Ajustes / tratamento dos dados (somente mortalidade)

def _norm_txt(txt: str) -> str:
    return (
        unicodedata.normalize("NFKD", str(txt))
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
    )

def _norm_col(col: str) -> str:
    return re.sub(r"\s+", "_", _norm_txt(col))

def _find_first_col(df: pd.DataFrame, candidates_norm: set[str]) -> str | None:
    for col in df.columns:
        if _norm_col(col) in candidates_norm:
            return col
    return None


def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    drop_cols = [c for c in df.columns if _norm_col(c).startswith("unnamed")]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

def _fmt_int(n):
    try:
        return f"{int(n):,}".replace(",", ".")
    except Exception:
        return str(n)

def _sum_row_wise(frame: pd.DataFrame, cols: list[str]) -> pd.Series:
    if not cols:
        return pd.Series([0] * len(frame), index=frame.index, dtype="int64")
    return (
        frame[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
        .round()
        .astype(int)
    )

def _col_ano_faixa(ano: int, faixa: str) -> str:
    return f"{int(ano)}_{faixa}"


def _col_total_ano(ano: int) -> str:
    return f"Total_{int(ano)}"


def _acumular_em_blocos(por_ano: pd.Series, anos_por_bloco: int) -> pd.Series:
    # Soma anos em blocos consecutivos e usa o último ano do bloco como rótulo.
    serie = pd.to_numeric(por_ano, errors="coerce").fillna(0).astype(int).sort_index()
    anos = serie.index.astype(int).tolist()

    acumulado = {}
    bloco_atual = []

    for ano in anos:
        bloco_atual.append(ano)
        if len(bloco_atual) == int(anos_por_bloco):
            ano_final = bloco_atual[-1]
            acumulado[ano_final] = int(serie.loc[bloco_atual].sum())
            bloco_atual = []

    # inclui bloco final incompleto para ir até o fim dos dados
    if bloco_atual:
        ano_final = bloco_atual[-1]
        acumulado[ano_final] = int(serie.loc[bloco_atual].sum())

    return pd.Series(acumulado).sort_index()


def aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
 
    df = _drop_index_cols(df)
    out = df.copy()

    col_mun = _find_first_col(out, {"municipios", "municipio", "cidades", "cidade"})
    col_cod = _find_first_col(out, {"codigo", "cod"})
    col_reg = _find_first_col(out, {"regional_de_saude", "regional_saude", "regional"})

    if MUNICIPIO is not None:
        if not col_mun:
            raise KeyError("Não encontrei coluna de município para filtrar.")
        out = out[out[col_mun].astype(str).map(_norm_txt) == _norm_txt(MUNICIPIO)]

    if CODIGO_MUNICIPIO is not None:
        if not col_cod:
            raise KeyError("Não encontrei coluna de código para filtrar.")
        out = out[pd.to_numeric(out[col_cod], errors="coerce").astype("Int64") == int(CODIGO_MUNICIPIO)]

    if REGIONAL_SAUDE is not None:
        if not col_reg:
            raise KeyError("Não encontrei coluna de regional para filtrar.")
        out = out[out[col_reg].astype(str).map(_norm_txt) == _norm_txt(REGIONAL_SAUDE)]

    return out


print("\n===================================================\n")
print("MUNICIPIO:", MUNICIPIO)
print("CODIGO_MUNICIPIO:", CODIGO_MUNICIPIO)
print("REGIONAL_SAUDE:", REGIONAL_SAUDE)
print("ANOS_RANGE_MORTALIDADE:", ANOS_RANGE_MORTALIDADE)
print("ANOS_POR_BLOCO_ACUMULADO:", ANOS_POR_BLOCO_ACUMULADO)

# Leitura e filtro
df_mort = pd.read_csv(CSV_MORTALIDADE)
f_mort = aplicar_filtros(df_mort)
print(f"Linhas selecionadas (mortalidade): {len(f_mort)}")

# Faixas e anos
ano_ini_m, ano_fim_m = map(int, ANOS_RANGE_MORTALIDADE)
anos_m = list(range(ano_ini_m, ano_fim_m + 1))

# 1) Barras: total por faixa etária no período
faixas_por_linha = pd.DataFrame(index=f_mort.index)
for faixa in FAIXAS_ETARIAS_MORTALIDADE:
    cols_faixa = [
        _col_ano_faixa(ano, faixa)
        for ano in anos_m
        if _col_ano_faixa(ano, faixa) in f_mort.columns
    ]
    faixas_por_linha[faixa] = _sum_row_wise(f_mort, cols_faixa)

por_faixa_mort = faixas_por_linha[FAIXAS_ETARIAS_MORTALIDADE].sum(axis=0).astype(int)

# 2) Pontos: total por ano
por_ano_mort = {}
for ano in anos_m:
    col_total = _col_total_ano(ano)
    if col_total in f_mort.columns:
        total_ano = pd.to_numeric(f_mort[col_total], errors="coerce").fillna(0).sum()
    else:
        cols_ano = [
            _col_ano_faixa(ano, faixa)
            for faixa in FAIXAS_ETARIAS_MORTALIDADE
            if _col_ano_faixa(ano, faixa) in f_mort.columns
        ]
        total_ano = _sum_row_wise(f_mort, cols_ano).sum()
    por_ano_mort[ano] = int(total_ano)

por_ano_mort_s = pd.Series(por_ano_mort).sort_index()
por_ano_mort_acum_s = _acumular_em_blocos(por_ano_mort_s, ANOS_POR_BLOCO_ACUMULADO)

print("Mortalidade — exemplo 40-49:", _fmt_int(int(por_faixa_mort.get("40-49", 0))))
if 2023 in por_ano_mort_s.index:
    print("Mortalidade — Total 2023:", _fmt_int(int(por_ano_mort_s.loc[2023])))
print("Blocos acumulados (últimos anos):", list(por_ano_mort_acum_s.index.astype(int)))

In [ ]:
# 4) Design e layout do gráfico de mortalidade 

def _xticks_fixos(x_min: int, x_max: int, passo: int = 500) -> list[int]:
    return list(range(int(x_min), int(x_max) + 1, int(passo)))

def plot_mortalidade(
    *,
    por_faixa: pd.Series,
    por_ano: pd.Series,
    titulo: str,
    anos_range: tuple[int, int],
    anos_passo: int = 1,
    yaxis2_title: str = "Ano",
    cor_barras: str = COR_BARRAS,
    cor_pontos: str = COR_PONTOS,
    width: int = LARGURA_GRAFICO,
    height: int = ALTURA_GRAFICO_MORTALIDADE,
    x_min: int = X_MIN,
    x_max: int = X_MAX,
):
    por_faixa = pd.to_numeric(por_faixa, errors="coerce").fillna(0)
    por_ano = pd.to_numeric(por_ano, errors="coerce").fillna(0)

    por_faixa = por_faixa[por_faixa > 0]
    por_ano = por_ano[por_ano > 0]
    if por_faixa.empty or por_ano.empty:
        raise ValueError("Sem dados > 0 para plotar.")

    categorias = por_faixa.index.astype(str).tolist()
    valores_barras = por_faixa.values.astype(int).tolist()

    anos_lista = por_ano.index.astype(int).tolist()
    valores_ano = por_ano.values.astype(int).tolist()

    ano_ini, ano_fim = map(int, anos_range)
    xticks = _xticks_fixos(x_min, x_max, passo=500)

    hover_barras = [
        f"<b>Faixa etária:</b> {cat}<br><b>Total:</b> {_fmt_int(val)}<br><i>Soma {ano_ini}–{ano_fim}</i>"
        for cat, val in zip(categorias, valores_barras)
    ]
    hover_pontos = [
        f"<b>{yaxis2_title}:</b> {ano}<br><b>Total:</b> {_fmt_int(val)}"
        for ano, val in zip(anos_lista, valores_ano)
    ]

    fig = go.Figure()

    fig.add_trace(go.Bar(
        name=f"Total por faixa etária (0 à 99+)",
        x=valores_barras,
        y=categorias,
        orientation="h",
        marker=dict(
            color=cor_barras,
            opacity=0.85,
            line=dict(color="white", width=1.0),
        ),
        text=[_fmt_int(v) for v in valores_barras],
        textposition="outside",
        textfont=dict(size=FONTE_TEXTO_BARRA, color="#222"),
        hovertext=hover_barras,
        hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
        yaxis="y",
        xaxis="x",
    ))

    fig.add_trace(go.Scatter(
        name=f"Total por {yaxis2_title.lower()} ({ano_ini}–{ano_fim})",
        x=valores_ano,
        y=anos_lista,
        mode="markers+text",
        marker=dict(
            color=cor_pontos,
            size=11,
            opacity=0.92,
            line=dict(color="#d73f3f"),
            symbol="circle",
        ),
        text=[_fmt_int(v) for v in valores_ano],
        textposition="middle right",
        textfont=dict(size=FONTE_TEXTO_PONTO, color=cor_pontos),
        hovertext=hover_pontos,
        hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
        yaxis="y2",
        xaxis="x",
    ))

    fig.update_layout(
        title=dict(
            text=titulo,
            font=dict(size=FONTE_TITULO, family="Arial", color="#1a1a2e"),
            x=0.5,
            xanchor="center",
            pad=dict(b=80),
        ),
        height=height,
        width=width,
        plot_bgcolor="#f8f9fb",
        paper_bgcolor="white",
        margin=dict(l=140, r=230, t=90, b=90),
        hovermode="closest",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.0,
            xanchor="center",
            x=0.5,
            font=dict(size=FONTE_LEGENDA),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#ddd",
            borderwidth=1,
        ),
        xaxis=dict(
            title="Número de mortes",
            tickvals=xticks,
            ticktext=[_fmt_int(v) for v in xticks],
            tickfont=dict(size=FONTE_EIXO_TICK),
            title_font=dict(size=FONTE_EIXO_TITULO),
            gridcolor="#dde3ed",
            gridwidth=1,
            showline=True,
            linecolor="#ccc",
            zeroline=True,
            zerolinecolor="#bbb",
            range=[x_min, x_max],
        ),
        yaxis=dict(
            title="Faixa etária",
            title_font=dict(size=FONTE_EIXO_TITULO),
            tickfont=dict(size=FONTE_EIXO_TICK),
            showgrid=False,
            showline=True,
            linecolor="#ccc",
            domain=[0, 1],
            type="category",
            categoryorder="array",
            categoryarray=categorias,
            showticklabels=True,
            automargin=True,
            tickmode="array",
            tickvals=categorias,
            ticktext=categorias,
        ),
        yaxis2=dict(
            title=yaxis2_title,
            title_font=dict(size=FONTE_EIXO_TITULO),
            tickfont=dict(size=FONTE_EIXO_TICK),
            overlaying="y",
            side="right",
            tickvals=anos_lista,
            showgrid=False,
            showline=True,
            linecolor="#ccc",
            range=[min(anos_lista) - 0.5, max(anos_lista) + 0.5],
        ),
        hoverlabel=dict(
            bgcolor="white",
            bordercolor="#aaa",
            font=dict(size=FONTE_EIXO_TICK, family="Arial"),
        ),
    )

    fig.update_layout(
        shapes=[
            dict(
                type="line",
                x0=0,
                x1=1,
                xref="paper",
                y0=cat,
                y1=cat,
                yref="y",
                line=dict(color="#dde3ed", width=1, dash="dot"),
            )
            for cat in categorias
        ]
    )

    fig.show()
    return fig


# Plot 1: Mortalidade anual
plot_mortalidade(
    por_faixa=por_faixa_mort.reindex(FAIXAS_ETARIAS_MORTALIDADE),
    por_ano=por_ano_mort_s,
    titulo=TITULO_MORTALIDADE,
    anos_range=ANOS_RANGE_MORTALIDADE,
    anos_passo=ANOS_PASSO_MORTALIDADE,
    yaxis2_title="Ano",
)

# Plot 2: Mortalidade acumulada por blocos
plot_mortalidade(
    por_faixa=por_faixa_mort.reindex(FAIXAS_ETARIAS_MORTALIDADE),
    por_ano=por_ano_mort_acum_s,
    titulo=f"{TITULO_MORTALIDADE_ACUM} (Acumulado {ANOS_POR_BLOCO_ACUMULADO} anos)",
    anos_range=ANOS_RANGE_MORTALIDADE,
    anos_passo=ANOS_POR_BLOCO_ACUMULADO,
    yaxis2_title="Fim do bloco",
)

print("Gráficos de mortalidade plotados (anual + acumulado)")

In [ ]:
# 5) Análises consolidadas com métricas já existentes (sem criar novas métricas-base)
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import re
import unicodedata

# ------------------------------
# Funções auxiliares de coluna
# ------------------------------
def _norm_txt_local(txt: str) -> str:
    return (
        unicodedata.normalize("NFKD", str(txt))
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
    )


def _norm_col_local(col: str) -> str:
    col = _norm_txt_local(col)
    col = re.sub(r"\s+", "_", col)
    return col


def _find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    norm_map = {_norm_col_local(c): c for c in df.columns}
    for cand in candidates:
        c_norm = _norm_col_local(cand)
        if c_norm in norm_map:
            return norm_map[c_norm]
    return None


# ------------------------------
# Base de dados consolidada
# ------------------------------
if "consolidado" in globals() and isinstance(consolidado, pd.DataFrame):
    base = consolidado.copy()
elif "metricas_mort" in globals() and isinstance(metricas_mort, pd.DataFrame):
    base = metricas_mort.copy()
elif "f_mort" in globals() and isinstance(f_mort, pd.DataFrame):
    print("Aviso: 'consolidado' e 'metricas_mort' não encontrados — construindo 'base' a partir de 'f_mort'.")
    df_tmp = f_mort.copy()

    col_cod_tmp = _find_col(df_tmp, ["codigo_municipio", "codigo", "cod_ibge", "cod"])
    col_mun_tmp = _find_col(df_tmp, ["nome_municipio", "municipio", "municipios", "cidade"])
    col_pop_tmp = _find_col(df_tmp, ["populacao_ano_referencia", "populacao_2022", "populacao", "população"]) 

    cols_total_ano = [c for c in df_tmp.columns if re.match(r"^Total_\d{4}$", str(c).strip())]

    if cols_total_ano:
        anual_num = df_tmp[cols_total_ano].apply(pd.to_numeric, errors="coerce").fillna(0)
    else:
        cols_ano_faixa = [c for c in df_tmp.columns if re.match(r"^\d{4}_.+", str(c).strip())]
        if cols_ano_faixa:
            anual_num = df_tmp[cols_ano_faixa].apply(pd.to_numeric, errors="coerce").fillna(0)
        else:
            numeric_cols = [c for c in df_tmp.columns if df_tmp[c].dtype.kind in "iuf"]
            if numeric_cols:
                anual_num = df_tmp[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
                print("Aviso: usei todas as colunas numéricas como base temporal (último recurso).")
            else:
                raise NameError(
                    "Não encontrei colunas suficientes em 'f_mort' para construir a base consolidada."
                )

    total_acumulado = anual_num.sum(axis=1)
    media_anual = anual_num.mean(axis=1)
    variancia_anual = anual_num.var(axis=1, ddof=0)
    desvio_padrao_anual = anual_num.std(axis=1, ddof=0)

    base = pd.DataFrame({
        "codigo_municipio": df_tmp[col_cod_tmp] if col_cod_tmp else pd.NA,
        "municipio": df_tmp[col_mun_tmp] if col_mun_tmp else df_tmp.index.astype(str),
        "total_mortes_acumulado": total_acumulado,
        "media_anual_mortes": media_anual,
        "variancia_anual": variancia_anual,
        "desvio_padrao_anual": desvio_padrao_anual,
    })

    if col_pop_tmp is not None:
        base["populacao"] = pd.to_numeric(df_tmp[col_pop_tmp], errors="coerce")
    else:
        base["populacao"] = np.nan
        print("Aviso: coluna de população não encontrada em 'f_mort'; percentuais podem ficar vazios.")
else:
    raise NameError("Não encontrei 'consolidado' nem 'metricas_mort' no notebook.")


# Detecta colunas equivalentes já existentes
col_cod = _find_col(base, ["codigo_municipio", "codigo ibge", "codigo", "cod_ibge", "cod_municipio"])
col_mun = _find_col(base, ["nome_municipio", "municipio", "municipios", "cidade"])
col_total = _find_col(base, ["total_mortes_acumulado", "valor absoluto (casos)", "valor_absoluto", "total_mortes", "total"])
col_media = _find_col(base, ["media_anual_mortes", "media anual (casos)", "media_anual", "media"])
col_var = _find_col(base, ["variancia_anual", "variancia anual", "variancia"])
col_dp = _find_col(base, ["desvio_padrao_anual", "desvio padrao anual", "desvio_padrao", "desvio padrao"])
col_pop = _find_col(base, ["populacao_ano_referencia", "populacao_2022", "populacao (2022)", "populacao"])
col_pct = _find_col(base, ["percentual_mortalidade_pop", "percentual casos/populacao (%)", "percentual", "percentual_populacao"])

obrigatorias = [
    ("Código", col_cod),
    ("Município", col_mun),
    ("Total acumulado", col_total),
    ("Média anual", col_media),
    ("Variância", col_var),
    ("Desvio padrão", col_dp),
]
faltantes = [nome for nome, col in obrigatorias if col is None]
if faltantes:
    raise KeyError(f"Colunas obrigatórias não encontradas em 'base': {faltantes}")

if col_pop is None:
    base["populacao_ausente"] = np.nan
    col_pop = "populacao_ausente"

# Padroniza tipos numéricos
for c in [col_total, col_media, col_var, col_dp, col_pop]:
    base[c] = pd.to_numeric(base[c], errors="coerce")

# Se a coluna percentual já existir, reaproveita; caso contrário calcula
if col_pct is None:
    base["percentual_calculado"] = np.where(base[col_pop] > 0, (base[col_total] / base[col_pop]) * 100, np.nan)
    col_pct = "percentual_calculado"
else:
    base[col_pct] = pd.to_numeric(base[col_pct], errors="coerce")

# ------------------------------
# Tabela principal (sem população e sem percentual)
# ------------------------------
tabela_exibir = pd.DataFrame({
    "Código IBGE": base[col_cod],
    "Município": base[col_mun],
    "Valor absoluto (casos)": base[col_total],
    "Média anual (casos)": base[col_media],
    "Variância anual": base[col_var],
    "Desvio padrão anual": base[col_dp],
}).copy()

tabela_exibir = tabela_exibir.sort_values("Valor absoluto (casos)", ascending=False, na_position="last").reset_index(drop=True)

total_casos = pd.to_numeric(tabela_exibir["Valor absoluto (casos)"], errors="coerce").fillna(0).sum()

linha_total = pd.DataFrame([
    {
        "Código IBGE": "",
        "Município": "TOTAL GERAL",
        "Valor absoluto (casos)": total_casos,
        "Média anual (casos)": pd.to_numeric(tabela_exibir["Média anual (casos)"], errors="coerce").fillna(0).sum(),
        "Variância anual": pd.to_numeric(tabela_exibir["Variância anual"], errors="coerce").fillna(0).sum(),
        "Desvio padrão anual": pd.to_numeric(tabela_exibir["Desvio padrão anual"], errors="coerce").fillna(0).sum(),
    }
])

tabela_exibir_total = pd.concat([linha_total, tabela_exibir], ignore_index=True)


def _destacar_total(row):
    if str(row.get("Município", "")).upper() == "TOTAL GERAL":
        return ["font-weight: 700; background-color: #eef3ff;"] * len(row)
    return [""] * len(row)


display(Markdown("### Tabela consolidada de mortalidade por município (MG)"))
display(
    tabela_exibir_total.style
    .format({
        "Código IBGE": "{}",
        "Valor absoluto (casos)": "{:,.0f}",
        "Média anual (casos)": "{:,.2f}",
        "Variância anual": "{:,.2f}",
        "Desvio padrão anual": "{:,.2f}",
    }, decimal=",", thousands=".")
    .set_table_styles([
        {"selector": "th", "props": "text-align: center; background-color: #f5f5f5;"},
        {"selector": "td", "props": "padding: 6px;"},
    ])
    .set_properties(**{"text-align": "left"}, subset=["Município"])
    .set_properties(**{"text-align": "center"}, subset=["Código IBGE"])
    .apply(_destacar_total, axis=1)
    .hide(axis="index")
)
print(f"Total de municípios na tabela (sem contar TOTAL GERAL): {len(tabela_exibir_total) - 1}")

In [ ]:
# 6) Dados numéricos: variação anual por município + rankings + série anual consolidada
import pandas as pd
import numpy as np
import re
import unicodedata
from IPython.display import display, Markdown

if "f_mort" not in globals() or not isinstance(f_mort, pd.DataFrame):
    raise NameError("DataFrame 'f_mort' não encontrado. Execute a célula de preparação dos dados primeiro.")

base_num = f_mort.copy()


def _norm_txt_num(txt: str) -> str:
    return (
        unicodedata.normalize("NFKD", str(txt))
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
    )

# Detecta coluna de município (robusto)
col_mun_num = None
for c in base_num.columns:
    cn = _norm_txt_num(c)
    if cn in {"municipio", "municipios", "cidade", "cidades", "nome_municipio"}:
        col_mun_num = c
        break

if col_mun_num is None:
    for c in base_num.columns:
        cn = _norm_txt_num(c)
        if "municip" in cn or "cidade" in cn:
            col_mun_num = c
            break

if col_mun_num is None:
    non_numeric = [c for c in base_num.columns if base_num[c].dtype.kind not in "iuf"]
    if non_numeric:
        col_mun_num = non_numeric[0]
        print(f"Aviso: coluna de município não identificada explicitamente; usando '{col_mun_num}' como rótulo de município.")

if col_mun_num is None:
    raise KeyError("Não encontrei coluna de município em f_mort.")

# Monta matriz anual por município
cols_total_ano = [c for c in base_num.columns if re.match(r"^Total_\d{4}$", str(c).strip())]

if cols_total_ano:
    anual = base_num[[col_mun_num] + cols_total_ano].copy()
    anual[cols_total_ano] = anual[cols_total_ano].apply(pd.to_numeric, errors="coerce").fillna(0)
    anual = anual.groupby(col_mun_num, as_index=True)[cols_total_ano].sum()
    anual.columns = [int(re.findall(r"\d{4}", c)[0]) for c in anual.columns]
    anual = anual.reindex(sorted(anual.columns), axis=1)
else:
    cols_ano_faixa = [c for c in base_num.columns if re.match(r"^\d{4}_.+", str(c).strip())]
    if not cols_ano_faixa:
        raise KeyError("Não encontrei colunas anuais (Total_YYYY ou YYYY_faixa) em f_mort.")

    anual_raw = base_num[[col_mun_num] + cols_ano_faixa].copy()
    anual_raw[cols_ano_faixa] = anual_raw[cols_ano_faixa].apply(pd.to_numeric, errors="coerce").fillna(0)
    anual_raw = anual_raw.groupby(col_mun_num, as_index=True)[cols_ano_faixa].sum()

    anos = sorted({int(str(c).split("_")[0]) for c in anual_raw.columns})
    anual = pd.DataFrame(index=anual_raw.index)
    for ano in anos:
        cols_ano = [c for c in anual_raw.columns if str(c).startswith(f"{ano}_")]
        anual[ano] = anual_raw[cols_ano].sum(axis=1)

anual = anual.sort_index(axis=1)

anos_ref = list(range(1979, 2025))
anos_existentes = [a for a in anos_ref if a in anual.columns]
if not anos_existentes:
    anos_existentes = anual.columns.tolist()
anual = anual[anos_existentes]

# Tabela anual consolidada (soma dos municípios por ano)
total_ano_s = anual.sum(axis=0).astype(float)
tabela_anual_estado = pd.DataFrame({
    "Ano": total_ano_s.index.astype(int),
    "Total de casos do ano": total_ano_s.values,
}).sort_values("Ano").reset_index(drop=True)

tabela_anual_estado["Valor referência (ano anterior)"] = tabela_anual_estado["Total de casos do ano"].shift(1)
if not tabela_anual_estado.empty:
    tabela_anual_estado.loc[0, "Valor referência (ano anterior)"] = tabela_anual_estado.loc[0, "Total de casos do ano"]

tabela_anual_estado["Resultado (abs)"] = (
    tabela_anual_estado["Total de casos do ano"] - tabela_anual_estado["Valor referência (ano anterior)"]
)
tabela_anual_estado["Resultado (%)"] = np.where(
    tabela_anual_estado["Valor referência (ano anterior)"] != 0,
    (tabela_anual_estado["Resultado (abs)"] / tabela_anual_estado["Valor referência (ano anterior)"]) * 100,
    np.nan
)
if not tabela_anual_estado.empty:
    tabela_anual_estado.loc[0, "Resultado (%)"] = 0.0

tabela_anual_estado["Situação"] = np.where(
    tabela_anual_estado["Resultado (abs)"] > 0,
    "Aumentou",
    np.where(tabela_anual_estado["Resultado (abs)"] < 0, "Diminuiu", "Sem variação")
)

# Tabela 1: variação em relação ao ano anterior por município
variacoes = []
for mun, row in anual.iterrows():
    prev = None
    for i, ano in enumerate(anual.columns):
        valor = float(row[ano]) if pd.notna(row[ano]) else 0.0
        if i == 0:
            ref = valor
            delta_abs = 0.0
            delta_pct = 0.0
        else:
            ref = float(prev) if prev is not None else 0.0
            delta_abs = valor - ref
            delta_pct = ((delta_abs / ref) * 100.0) if ref != 0 else np.nan
        variacoes.append({
            "Município": mun,
            "Ano": int(ano),
            "Casos": valor,
            "Referência ano anterior": ref,
            "Aumento/queda (abs)": delta_abs,
            "Aumento/queda (%)": delta_pct,
        })
        prev = valor

tabela_variacao_anual = pd.DataFrame(variacoes)

# Tabela 2: resumo por município (mantida)
resumo = pd.DataFrame(index=anual.index)
resumo["Total de casos"] = anual.sum(axis=1)
resumo["Média anual"] = anual.mean(axis=1)
resumo["Desvio padrão"] = anual.std(axis=1, ddof=0)

max_aumento_abs = tabela_variacao_anual.groupby("Município")["Aumento/queda (abs)"].max()
min_aumento_pct = tabela_variacao_anual.groupby("Município")["Aumento/queda (%)"].min()

resumo["Maior aumento (abs)"] = max_aumento_abs
resumo["Menor aumento (%)"] = min_aumento_pct
resumo_indicadores = resumo.reset_index()
if resumo_indicadores.columns[0] != "Município":
    resumo_indicadores = resumo_indicadores.rename(columns={resumo_indicadores.columns[0]: "Município"})

# Municípios sem mortalidade no período:
# referência = tabela de municípios de MG (853)
CSV_MUNICIPIOS_REF = "/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv"
try:
    ref_mun_df = pd.read_csv(CSV_MUNICIPIOS_REF)
    if "municipio_original" in ref_mun_df.columns:
        col_ref_mun = "municipio_original"
    elif "municipio_normalizado" in ref_mun_df.columns:
        col_ref_mun = "municipio_normalizado"
    else:
        col_ref_mun = ref_mun_df.columns[0]
        print(f"Aviso: coluna de município não encontrada explicitamente na referência; usando '{col_ref_mun}'.")
except Exception as e:
    print(f"Aviso: não foi possível ler '{CSV_MUNICIPIOS_REF}' ({e}). Usando municípios da base de mortalidade.")
    ref_mun_df = base_num[[col_mun_num]].rename(columns={col_mun_num: "municipio_original"})
    col_ref_mun = "municipio_original"

mun_ref_df = ref_mun_df[[col_ref_mun]].dropna().copy()
mun_ref_df["_mun_norm"] = mun_ref_df[col_ref_mun].astype(str).map(_norm_txt_num)
mun_ref_df = mun_ref_df[mun_ref_df["_mun_norm"] != ""]

mun_ref_map = (
    mun_ref_df.drop_duplicates("_mun_norm")
    .set_index("_mun_norm")[col_ref_mun]
    .astype(str)
    .to_dict()
)

mun_com_morte_norm = set(
    resumo_indicadores.loc[resumo_indicadores["Total de casos"] > 0, "Município"]
    .astype(str)
    .map(_norm_txt_num)
    .tolist()
)

mun_ref_norm = set(mun_ref_map.keys())
mun_sem_morte_norm = sorted(mun_ref_norm - mun_com_morte_norm)
cidades_zero_mortalidade = sorted([str(mun_ref_map[n]) for n in mun_sem_morte_norm])

# Exibições principais
display(Markdown("### Tabela 1 — Série anual consolidada (soma dos municípios)"))
display(
    tabela_anual_estado.style.format({
        "Ano": "{:,.0f}",
        "Total de casos do ano": "{:,.0f}",
        "Valor referência (ano anterior)": "{:,.0f}",
        "Resultado (abs)": "{:+,.0f}",
        "Resultado (%)": "{:+,.2f}",
    }, decimal=",", thousands=".")
)

# display(Markdown("### Tabela 2 — Indicadores por município"))
# display(
#     resumo_indicadores
#     .sort_values("Total de casos", ascending=False)
#     .reset_index(drop=True)
#     .style
#     .format({
#         "Total de casos": "{:,.0f}",
#         "Média anual": "{:,.2f}",
#         "Desvio padrão": "{:,.2f}",
#         "Maior aumento (abs)": "{:+,.0f}",
#         "Menor aumento (%)": "{:+,.2f}",
#     }, decimal=",", thousands=".")
# )

# Rankings em formato DataFrame
rank_maior_total = resumo_indicadores.nlargest(3, "Total de casos")[["Município", "Total de casos"]]
rank_maior_media = resumo_indicadores.nlargest(3, "Média anual")[["Município", "Média anual"]]
rank_maior_dp = resumo_indicadores.nlargest(3, "Desvio padrão")[["Município", "Desvio padrão"]]
rank_maior_aumento = resumo_indicadores.nlargest(3, "Maior aumento (abs)")[["Município", "Maior aumento (abs)"]]

rank_menor_total = resumo_indicadores.nsmallest(3, "Total de casos")[["Município", "Total de casos"]]
rank_menor_media = resumo_indicadores.nsmallest(3, "Média anual")[["Município", "Média anual"]]
rank_menor_dp = resumo_indicadores.nsmallest(3, "Desvio padrão")[["Município", "Desvio padrão"]]
rank_menor_aumento = resumo_indicadores.nsmallest(3, "Menor aumento (%)")[["Município", "Menor aumento (%)"]]

display(Markdown("### Top 3 — maior número de casos"))
display(rank_maior_total.style.format({"Total de casos": "{:,.0f}"}, decimal=",", thousands="."))

display(Markdown("### Top 3 — maior média anual"))
display(rank_maior_media.style.format({"Média anual": "{:,.2f}"}, decimal=",", thousands="."))

display(Markdown("### Top 3 — maior desvio padrão"))
display(rank_maior_dp.style.format({"Desvio padrão": "{:,.2f}"}, decimal=",", thousands="."))

display(Markdown("### Top 3 — maior aumento (abs)"))
display(rank_maior_aumento.style.format({"Maior aumento (abs)": "{:+,.0f}"}, decimal=",", thousands="."))

display(Markdown("### Top 3 — menor número de casos"))
display(rank_menor_total.style.format({"Total de casos": "{:,.0f}"}, decimal=",", thousands="."))

display(Markdown("### Top 3 — menor média anual"))
display(rank_menor_media.style.format({"Média anual": "{:,.2f}"}, decimal=",", thousands="."))

display(Markdown("### Top 3 — menor desvio padrão"))
display(rank_menor_dp.style.format({"Desvio padrão": "{:,.2f}"}, decimal=",", thousands="."))

display(Markdown("### Top 3 — menor aumento (%)"))
display(rank_menor_aumento.style.format({"Menor aumento (%)": "{:+,.2f}"}, decimal=",", thousands="."))

# Lista de municípios sem mortalidade em DataFrame
df_cidades_zero = pd.DataFrame({"Município sem mortalidade": cidades_zero_mortalidade})
display(Markdown("### Municípios sem mortalidade no período"))
if df_cidades_zero.empty:
    print("Nenhum município com mortalidade zero no período.")
else:
    display(df_cidades_zero)

In [ ]:
# 7) Mapa de MG com destaque para municípios sem mortalidade no período
import geobr
import matplotlib.pyplot as plt
import unicodedata
import pandas as pd
from matplotlib.patches import Patch
from IPython.display import display, Markdown

if "cidades_zero_mortalidade" not in globals():
    raise NameError("Lista 'cidades_zero_mortalidade' não encontrada. Execute primeiro a célula 6.")
if "resumo_indicadores" not in globals():
    raise NameError("DataFrame 'resumo_indicadores' não encontrado. Execute primeiro a célula 6.")


def _norm_nome_muni(x: str) -> str:
    return (
        unicodedata.normalize("NFKD", str(x))
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
    )

# conjunto normalizado de municípios sem mortalidade no período
sem_morte_total_norm = {_norm_nome_muni(c) for c in cidades_zero_mortalidade}

# imprime nomes das cidades sem mortes
df_sem_mortes = pd.DataFrame({"Municípios sem mortalidade no período": sorted(cidades_zero_mortalidade)})
display(Markdown("### Municípios sem mortalidade no período"))
display(df_sem_mortes)

# malha municipal de MG
mg_muni = geobr.read_municipality(code_muni="MG", year=2020)

# nome da coluna de município na malha
col_nome_mg = None
for cand in ["name_muni", "nome_municipio", "municipio", "nome"]:
    if cand in mg_muni.columns:
        col_nome_mg = cand
        break

if col_nome_mg is None:
    raise KeyError(f"Não encontrei coluna de nome de município na malha do geobr. Colunas: {list(mg_muni.columns)}")

# classe no mapa
mg_muni["nome_norm"] = mg_muni[col_nome_mg].astype(str).map(_norm_nome_muni)
mg_muni["sem_mortalidade_periodo"] = mg_muni["nome_norm"].isin(sem_morte_total_norm)

fig, ax = plt.subplots(figsize=(12, 12))

# base (linhas de separação mais escuras no mapa todo)
mg_muni.plot(ax=ax, color="#e8edf3", edgecolor="#5b6470", linewidth=0.6)

# sem mortalidade no período (vermelho) com contorno ainda mais escuro
mg_muni[mg_muni["sem_mortalidade_periodo"]].plot(
    ax=ax, color="#d73f3f", edgecolor="#111111", linewidth=1.35
)

ax.set_title("Minas Gerais — municípios sem mortalidade no período", fontsize=16, pad=18)
ax.axis("off")

legend_handles = [
    Patch(facecolor="#d73f3f", edgecolor="#111111", label="Sem mortalidade no período"),
]
ax.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.01),
    frameon=True,
    ncol=1,
)

plt.tight_layout()
plt.show()

qtd_sem_morte_periodo = int(mg_muni["sem_mortalidade_periodo"].sum())
print(f"Sem mortalidade no período ({qtd_sem_morte_periodo}, vermelho): conjunto total de municípios sem mortes no período.")

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import plotly.express as px
import geobr
from IPython.display import display

# =========================
# Parâmetros (mortalidade)
# =========================
CSV_POPULACAO_FAIXA_ETARIA = globals().get(
    "CSV_POPULACAO_FAIXA_ETARIA",
    "/home/anna/tcc/Tratamento/População - Faixa Etária/tabela_populacao_faixa_etaria.csv",
)
ANO_REFERENCIA_POPULACAO_MORTALIDADE = int(globals().get("ANO_REFERENCIA_POPULACAO_MORTALIDADE", 2022))
FILTRO_FAIXA_ETARIA_MORTALIDADE = globals().get("FILTRO_FAIXA_ETARIA_MORTALIDADE", None)
ORDENAR_TABELA_POR_MORTALIDADE = globals().get("ORDENAR_TABELA_POR_MORTALIDADE", "valor_absoluto_mortes")

if "ANOS_RANGE_MORTALIDADE" in globals() and isinstance(ANOS_RANGE_MORTALIDADE, tuple):
    DIVISOR_MEDIA_ANUAL_MORTALIDADE = int(globals().get(
        "DIVISOR_MEDIA_ANUAL_MORTALIDADE",
        int(ANOS_RANGE_MORTALIDADE[1]) - int(ANOS_RANGE_MORTALIDADE[0]) + 1
    ))
else:
    DIVISOR_MEDIA_ANUAL_MORTALIDADE = int(globals().get("DIVISOR_MEDIA_ANUAL_MORTALIDADE", 45))

# =========================
# Funções auxiliares
# =========================
def _normalizar_codigo_municipio(serie):
    s = pd.to_numeric(serie, errors="coerce")
    return s.round().astype("Int64").astype(str).replace("<NA>", pd.NA).str.zfill(7)


def _peso_faixa_mort(df_segmento, filtro_faixa=None):
    if "_drop_index_cols" in globals():
        base = _drop_index_cols(df_segmento)
    else:
        base = df_segmento.copy()

    if "_find_first_col" in globals():
        col_mun = _find_first_col(base, {"municipios", "municipio", "cidades", "cidade", "nome_municipio"})
        col_cod = _find_first_col(base, {"codigo_municipio", "cod_municipio", "codigo", "cod", "codigo_ibge"})
    else:
        col_mun, col_cod = None, None

    if col_mun is None:
        candidatos = [c for c in base.columns if "mun" in str(c).lower() or "cidade" in str(c).lower()]
        col_mun = candidatos[0] if candidatos else None
    if col_cod is None:
        candidatos = [c for c in base.columns if "cod" in str(c).lower()]
        col_cod = candidatos[0] if candidatos else None

    if col_mun is None or col_cod is None:
        raise KeyError("Não foi possível identificar colunas de município/código na base de mortalidade.")

    if filtro_faixa is None:
        filtro = None
    elif isinstance(filtro_faixa, (list, tuple, set)):
        if "_norm_txt" in globals():
            filtro = {_norm_txt(v) for v in filtro_faixa}
        else:
            filtro = {str(v).strip().lower() for v in filtro_faixa}
    else:
        if "_norm_txt" in globals():
            filtro = {_norm_txt(filtro_faixa)}
        else:
            filtro = {str(filtro_faixa).strip().lower()}

    rows = []
    for col in base.columns:
        txt = str(col)
        if "_" not in txt:
            continue
        ano_str, cat = txt.split("_", 1)
        if not ano_str.isdigit():
            continue
        cat_norm = _norm_txt(cat) if "_norm_txt" in globals() else str(cat).strip().lower()
        rows.append(
            pd.DataFrame({
                "codigo_municipio": _normalizar_codigo_municipio(base[col_cod]),
                "nome_municipio": base[col_mun].astype(str).str.strip(),
                "ano": int(ano_str),
                "valor": pd.to_numeric(base[col], errors="coerce").fillna(0),
                "sel": True if filtro is None else (cat_norm in filtro),
            })
        )

    if not rows:
        return pd.DataFrame(columns=["codigo_municipio", "nome_municipio", "ano", "peso_faixa"])

    long = pd.concat(rows, ignore_index=True)
    tot = long.groupby(["codigo_municipio", "nome_municipio", "ano"], as_index=False)["valor"].sum()
    sel = (
        long[long["sel"]]
        .groupby(["codigo_municipio", "nome_municipio", "ano"], as_index=False)["valor"]
        .sum()
        .rename(columns={"valor": "valor_sel"})
    )
    out = tot.merge(sel, on=["codigo_municipio", "nome_municipio", "ano"], how="left")
    out["valor_sel"] = pd.to_numeric(out["valor_sel"], errors="coerce").fillna(0)
    out["peso_faixa"] = 1.0 if filtro is None else out["valor_sel"].div(out["valor"].replace(0, pd.NA)).fillna(0.0)
    return out[["codigo_municipio", "nome_municipio", "ano", "peso_faixa"]]


def _classe_1a10(s):
    bins = [-float("inf"), 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, float("inf")]
    labels = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10+"]
    return pd.cut(pd.to_numeric(s, errors="coerce").fillna(0), bins=bins, labels=labels, right=True).astype(str)

# =========================
# Base anual de mortalidade
# =========================
if "CSV_MORTALIDADE" not in globals():
    raise NameError("CSV_MORTALIDADE não definido no notebook.")

df_mort_raw = pd.read_csv(CSV_MORTALIDADE)
df_mort = aplicar_filtros(df_mort_raw) if "aplicar_filtros" in globals() else df_mort_raw.copy()

if "_drop_index_cols" in globals():
    df_mort = _drop_index_cols(df_mort)

if "_find_first_col" in globals():
    col_mun = _find_first_col(df_mort, {"municipios", "municipio", "cidades", "cidade", "nome_municipio"})
    col_cod = _find_first_col(df_mort, {"codigo_municipio", "cod_municipio", "codigo", "cod", "codigo_ibge"})
else:
    col_mun, col_cod = None, None

if col_mun is None:
    candidatos = [c for c in df_mort.columns if "mun" in str(c).lower() or "cidade" in str(c).lower()]
    col_mun = candidatos[0] if candidatos else None
if col_cod is None:
    candidatos = [c for c in df_mort.columns if "cod" in str(c).lower()]
    col_cod = candidatos[0] if candidatos else None

if col_mun is None or col_cod is None:
    raise KeyError("Não foi possível identificar as colunas de município/código em CSV_MORTALIDADE.")

cols_total_ano = [c for c in df_mort.columns if re.match(r"^Total_\d{4}$", str(c).strip())]
rows = []

if cols_total_ano:
    for c in cols_total_ano:
        ano = int(str(c).split("_")[1])
        rows.append(
            pd.DataFrame({
                "codigo_municipio": _normalizar_codigo_municipio(df_mort[col_cod]),
                "nome_municipio": df_mort[col_mun].astype(str).str.strip(),
                "ano": ano,
                "total_mortes_ano": pd.to_numeric(df_mort[c], errors="coerce").fillna(0),
            })
        )
else:
    cols_ano_faixa = [c for c in df_mort.columns if re.match(r"^\d{4}_.+", str(c).strip())]
    if not cols_ano_faixa:
        raise KeyError("Não encontrei colunas anuais em CSV_MORTALIDADE (Total_YYYY ou YYYY_faixa).")

    tmp = df_mort[[col_cod, col_mun] + cols_ano_faixa].copy()
    tmp[cols_ano_faixa] = tmp[cols_ano_faixa].apply(pd.to_numeric, errors="coerce").fillna(0)
    anos = sorted({int(str(c).split("_")[0]) for c in cols_ano_faixa})
    for ano in anos:
        cols_ano = [c for c in cols_ano_faixa if str(c).startswith(f"{ano}_")]
        rows.append(
            pd.DataFrame({
                "codigo_municipio": _normalizar_codigo_municipio(tmp[col_cod]),
                "nome_municipio": tmp[col_mun].astype(str).str.strip(),
                "ano": ano,
                "total_mortes_ano": tmp[cols_ano].sum(axis=1),
            })
        )

base_ano = (
    pd.concat(rows, ignore_index=True)
    .groupby(["codigo_municipio", "nome_municipio", "ano"], as_index=False)["total_mortes_ano"]
    .sum()
)

peso = _peso_faixa_mort(df_mort_raw, FILTRO_FAIXA_ETARIA_MORTALIDADE)
mortes_filtradas = base_ano.merge(peso, on=["codigo_municipio", "nome_municipio", "ano"], how="left")
mortes_filtradas["peso_faixa"] = pd.to_numeric(mortes_filtradas["peso_faixa"], errors="coerce").fillna(1.0)
mortes_filtradas["mortes_ajustadas_ano"] = mortes_filtradas["total_mortes_ano"] * mortes_filtradas["peso_faixa"]

municipios = mortes_filtradas.groupby(["codigo_municipio", "nome_municipio"], as_index=False).agg(
    valor_absoluto_mortes=("mortes_ajustadas_ano", "sum"),
    variancia_anual=("mortes_ajustadas_ano", lambda s: float(pd.Series(s).var(ddof=0))),
    desvio_padrao_anual=("mortes_ajustadas_ano", lambda s: float(pd.Series(s).std(ddof=0))),
)
municipios["media_anual_mortes"] = municipios["valor_absoluto_mortes"] / float(DIVISOR_MEDIA_ANUAL_MORTALIDADE)

# =========================
# População de referência
# =========================
pop = pd.read_csv(CSV_POPULACAO_FAIXA_ETARIA)
if "_drop_index_cols" in globals():
    pop = _drop_index_cols(pop)

if "_find_first_col" in globals():
    pm = _find_first_col(pop, {"municipios", "municipio", "cidades", "cidade", "nome_municipio"})
    pc = _find_first_col(pop, {"codigo_municipio", "cod_municipio", "codigo", "cod", "codigo_ibge"})
else:
    pm, pc = None, None

if pm is None:
    candidatos = [c for c in pop.columns if "mun" in str(c).lower() or "cidade" in str(c).lower()]
    pm = candidatos[0] if candidatos else None
if pc is None:
    candidatos = [c for c in pop.columns if "cod" in str(c).lower()]
    pc = candidatos[0] if candidatos else None

if pm is None or pc is None:
    raise KeyError("Não foi possível identificar colunas de município/código em CSV_POPULACAO_FAIXA_ETARIA.")

pcols = [c for c in pop.columns if str(c).startswith(f"{ANO_REFERENCIA_POPULACAO_MORTALIDADE}_")]
if not pcols:
    raise KeyError(
        f"Não encontrei colunas de população para o ano {ANO_REFERENCIA_POPULACAO_MORTALIDADE} em CSV_POPULACAO_FAIXA_ETARIA."
    )

pop["populacao_ano_referencia"] = pop[pcols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
pop_ref = (
    pop.assign(
        codigo_municipio=_normalizar_codigo_municipio(pop[pc]),
        nome_municipio=pop[pm].astype(str).str.strip(),
    )
    .groupby(["codigo_municipio", "nome_municipio"], as_index=False)["populacao_ano_referencia"]
    .sum()
)

consolidado_mortalidade = municipios.merge(
    pop_ref[["codigo_municipio", "populacao_ano_referencia"]],
    on="codigo_municipio",
    how="left",
)
consolidado_mortalidade["populacao_ano_referencia"] = pd.to_numeric(
    consolidado_mortalidade["populacao_ano_referencia"], errors="coerce"
).fillna(0)
consolidado_mortalidade["percentual_mortes_pop"] = (
    consolidado_mortalidade["valor_absoluto_mortes"]
    .div(consolidado_mortalidade["populacao_ano_referencia"].replace(0, pd.NA))
    .fillna(0) * 100
)

consolidado_mortalidade["escala_media_1a10+"] = _classe_1a10(consolidado_mortalidade["media_anual_mortes"])
consolidado_mortalidade["escala_percentual_1a10+"] = _classe_1a10(consolidado_mortalidade["percentual_mortes_pop"])

# =========================
# Mapas MG (Plotly)
# =========================
mg = geobr.read_municipality(code_muni="MG", year=ANO_MALHA_MUNICIPIOS_MG).to_crs(4326)
mg["codigo_municipio"] = _normalizar_codigo_municipio(mg["code_muni"])
geojson_mg = json.loads(mg[["codigo_municipio", "geometry"]].to_json())

mapa = mg[["codigo_municipio"]].merge(consolidado_mortalidade, on="codigo_municipio", how="left")
cores = {
    "0": "#f7fbff",
    "1": "#deebf7",
    "2": "#c6dbef",
    "3": "#9ecae1",
    "4": "#6baed6",
    "5": "#4292c6",
    "6": "#2171b5",
    "7": "#08519c",
    "8": "#fcbba1",
    "9": "#fb6a4a",
    "10+": "#cb181d",
}


def _plot_mort(df, col_val, col_classe, titulo, legenda):
    fig = px.choropleth(
        df,
        geojson=geojson_mg,
        locations="codigo_municipio",
        featureidkey="properties.codigo_municipio",
        color=col_classe,
        color_discrete_map=cores,
        category_orders={col_classe: ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10+"]},
        hover_name="nome_municipio",
        hover_data={
            "codigo_municipio": True,
            col_val: ":.2f",
            "valor_absoluto_mortes": ":.2f",
            "percentual_mortes_pop": ":.2f",
            "media_anual_mortes": ":.2f",
        },
        labels={col_classe: legenda},
        title=titulo,
    )
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        showcountries=False,
        showcoastlines=False,
        showland=False,
        showframe=False,
    )
    fig.update_traces(marker_line_color="#566173", marker_line_width=0.7)
    fig.update_layout(
        template="plotly_white",
        width=1250,
        height=820,
        margin=dict(l=10, r=10, t=80, b=10),
        title=dict(x=0.5, xanchor="center"),
        legend=dict(orientation="h", yanchor="top", y=0.98, xanchor="center", x=0.5),
    )
    display(fig)


_plot_mort(
    mapa,
    "media_anual_mortes",
    "escala_media_1a10+",
    "MG — Média anual de mortes por município (faixa etária)",
    "Escala (1 a 10+) - média anual",
)
_plot_mort(
    mapa,
    "percentual_mortes_pop",
    "escala_percentual_1a10+",
    f"MG — Percentual de mortes em relação à população ({ANO_REFERENCIA_POPULACAO_MORTALIDADE}) (faixa etária)",
    "Escala (1 a 10+) - percentual",
)

# =========================
# Tabela final (DataFrame)
# =========================
tabela_final = consolidado_mortalidade[[
    "codigo_municipio",
    "nome_municipio",
    "valor_absoluto_mortes",
    "media_anual_mortes",
    "variancia_anual",
    "desvio_padrao_anual",
    "populacao_ano_referencia",
    "percentual_mortes_pop",
]].copy()

if ORDENAR_TABELA_POR_MORTALIDADE in tabela_final.columns:
    tabela_final = tabela_final.sort_values(ORDENAR_TABELA_POR_MORTALIDADE, ascending=False).reset_index(drop=True)

tabela_design = tabela_final.rename(columns={
    "codigo_municipio": "Código IBGE",
    "nome_municipio": "Município",
    "valor_absoluto_mortes": "Valor absoluto (mortes)",
    "media_anual_mortes": "Média anual (mortes)",
    "variancia_anual": "Variância anual",
    "desvio_padrao_anual": "Desvio padrão anual",
    "populacao_ano_referencia": f"População ({ANO_REFERENCIA_POPULACAO_MORTALIDADE})",
    "percentual_mortes_pop": "Percentual mortes/população (%)",
})

for col in [
    "Valor absoluto (mortes)",
    "Média anual (mortes)",
    "Variância anual",
    "Desvio padrão anual",
    "Percentual mortes/população (%)",
]:
    tabela_design[col] = pd.to_numeric(tabela_design[col], errors="coerce").round(2)

tabela_design[f"População ({ANO_REFERENCIA_POPULACAO_MORTALIDADE})"] = pd.to_numeric(
    tabela_design[f"População ({ANO_REFERENCIA_POPULACAO_MORTALIDADE})"],
    errors="coerce",
).fillna(0).astype(int)

display(tabela_design)
print("✅ Mapas MG e tabela de mortalidade gerados")

In [ ]:
# Mapa acumulado 1979–2024: MUNICÍPIOS
import json
import re
import numpy as np
import pandas as pd
import plotly.express as px
import geobr

ANO_INI_MAPA = 1979
ANO_FIM_MAPA = 2024
COR_SEM_MORTALIDADE = "#6ca8e5"
FAIXAS_MUNICIPIOS = [50, 100, 500, 1000, 5000]
# PALETA_MAPA = ["#ffffd4", "#fec44f", "#ee9c21", "#ff7f50", "#d20000", "#9c2328"]
PALETA_MAPA = ["#9c2328", "#d20000", "#ff7f50", "#ee9c21", "#fec44f", "#ffffd4"]

def _norm_cod7(s):
    x = pd.to_numeric(s, errors="coerce")
    return x.round().astype("Int64").astype(str).replace("<NA>", pd.NA).str.zfill(7)

def _first_col(df, candidatos):
    cols_norm = {str(c).strip().lower(): c for c in df.columns}
    for c in candidatos:
        if c.lower() in cols_norm:
            return cols_norm[c.lower()]
    return None

def _rotular_faixas_fixas(serie, faixas):
    s = pd.to_numeric(serie, errors="coerce").fillna(0)
    labels = ["Sem mortalidade (0)"]
    inicio = 1
    for i, limite in enumerate(faixas):
        if i == 0:
            labels.append(f"{inicio}–{limite}")
        else:
            labels.append(f"{faixas[i-1]+1:,}–{limite:,}".replace(",", "."))
        inicio = limite + 1
    labels.append(f"{faixas[-1]+1:,}+".replace(",", "."))
    classe = pd.Series("Sem mortalidade (0)", index=s.index)
    if (s > 0).any():
        inicio = 1
        idx_faixa = 1
        for limite in faixas:
            mask = (s >= inicio) & (s <= limite)
            classe.loc[mask] = labels[idx_faixa]
            inicio = limite + 1
            idx_faixa += 1
        classe.loc[s > faixas[-1]] = labels[-1]
    labels_usados = ["Sem mortalidade (0)"] + [lab for lab in labels[1:] if (classe == lab).any()]
    return classe, labels_usados

# Carrega base
if "f_mort" in globals() and isinstance(f_mort, pd.DataFrame):
    base_mort = f_mort.copy()
elif "CSV_MORTALIDADE" in globals():
    raw_mort = pd.read_csv(CSV_MORTALIDADE)
    base_mort = aplicar_filtros(raw_mort) if "aplicar_filtros" in globals() else raw_mort
else:
    csv_mort_padrao = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"
    raw_mort = pd.read_csv(csv_mort_padrao)
    base_mort = aplicar_filtros(raw_mort) if "aplicar_filtros" in globals() else raw_mort

if "_drop_index_cols" in globals():
    base_mort = _drop_index_cols(base_mort)

col_cod = _first_col(base_mort, {"Código","Codigo","codigo_municipio","cod_municipio","codigo","cod","codigo_ibge"})
col_mun = _first_col(base_mort, {"Municípios","Municipios","Município","Municipio","municipios","municipio","cidades","cidade","nome_municipio"})
col_reg = _first_col(base_mort, {"Regional de Saúde","Regional de Saude","Regional","regional_de_saude","regional_saude","regional"})

if col_cod is None or col_mun is None or col_reg is None:
    raise KeyError(f"Colunas não encontradas: cod={col_cod}, mun={col_mun}, reg={col_reg}\n{base_mort.columns.tolist()}")

anos = list(range(ANO_INI_MAPA, ANO_FIM_MAPA + 1))
cols_total = [f"Total_{a}" for a in anos if f"Total_{a}" in base_mort.columns]
if cols_total:
    acumulado_linha = base_mort[cols_total].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
else:
    cols_faixa = [c for c in base_mort.columns
                  if re.match(r"^\d{4}_.+", str(c).strip()) and int(str(c).split("_",1)[0]) in anos]
    acumulado_linha = base_mort[cols_faixa].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

base_acum = pd.DataFrame({
    "codigo_municipio": _norm_cod7(base_mort[col_cod]),
    "nome_municipio":   base_mort[col_mun].astype(str).str.strip(),
    "mortes_1979_2024": acumulado_linha,
})

por_municipio = (
    base_acum.groupby(["codigo_municipio","nome_municipio"], as_index=False)["mortes_1979_2024"].sum()
)

ano_malha = int(globals().get("ANO_MALHA_MUNICIPIOS_MG", 2020))
mg = geobr.read_municipality(code_muni="MG", year=ano_malha).to_crs(4326)
mg["codigo_municipio"] = _norm_cod7(mg["code_muni"])

mapa_mun = mg[["codigo_municipio","geometry"]].merge(por_municipio, on="codigo_municipio", how="left")
mapa_mun["nome_municipio"]   = mapa_mun["nome_municipio"].fillna("")
mapa_mun["mortes_1979_2024"] = pd.to_numeric(mapa_mun["mortes_1979_2024"], errors="coerce").fillna(0)

classe_mun, ordem_mun = _rotular_faixas_fixas(mapa_mun["mortes_1979_2024"], FAIXAS_MUNICIPIOS)
mapa_mun["classe_mortes"] = classe_mun

cores_mun = {"Sem mortalidade (0)": COR_SEM_MORTALIDADE}
labels_positivos_mun = [x for x in ordem_mun if x != "Sem mortalidade (0)"]
for i, lab in enumerate(reversed(labels_positivos_mun)):
    cores_mun[lab] = PALETA_MAPA[min(i, len(PALETA_MAPA)-1)]

geojson_mun = json.loads(mapa_mun[["codigo_municipio","geometry"]].to_json())

fig_mun = px.choropleth(
    mapa_mun,
    geojson=geojson_mun,
    locations="codigo_municipio",
    featureidkey="properties.codigo_municipio",
    color="classe_mortes",
    color_discrete_map=cores_mun,
    category_orders={"classe_mortes": ordem_mun},
    hover_name="nome_municipio",
    hover_data={"codigo_municipio": True, "mortes_1979_2024": ":,.0f", "classe_mortes": True},
    labels={"classe_mortes": "Mortalidade (1979–2024)", "mortes_1979_2024": "Total de mortes"},
    title="Mortalidade acumulada por município (1979–2024)",
)
fig_mun.update_geos(fitbounds="locations", visible=False,
                    showcountries=False, showcoastlines=False,
                    showland=False, showframe=False)
# width=0 evita as pintinhas nas bordas dos 853 municípios
# Linhas escuras entre municípios:
fig_mun.update_traces(marker_line_color="#2d3436", marker_line_width=0.8)
fig_mun.update_layout(
    template="plotly_white",
    width=1250, height=820,
    margin=dict(l=10, r=10, t=80, b=10),
    title=dict(x=0.5, xanchor="center", font=dict(size=20)),
    legend=dict(orientation="h", yanchor="top", y=0.99,
                xanchor="center", x=0.5, title=dict(text="")),
)
fig_mun.show()
print("✅ Mapa por município gerado")

In [ ]:
# Mapa acumulado 1979–2024: REGIONAIS DE SAÚDE
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import plotly.express as px
import geobr

ANO_INI_MAPA = 1979
ANO_FIM_MAPA = 2024
FAIXAS_REGIONAIS = [300, 500, 1000, 3000, 5000, 15000]
PALETA_MAPA = ["#9c2328", "#d20000", "#ff7f50", "#fec44f", "#ee9c21", "#ffffd4"]

def _norm_cod7(s):
    x = pd.to_numeric(s, errors="coerce")
    return x.round().astype("Int64").astype(str).replace("<NA>", pd.NA).str.zfill(7)

def _first_col(df, candidatos):
    cols_norm = {str(c).strip().lower(): c for c in df.columns}
    for c in candidatos:
        if c.lower() in cols_norm:
            return cols_norm[c.lower()]
    return None

def _rotular_faixas_fixas(serie, faixas):
    s = pd.to_numeric(serie, errors="coerce").fillna(0)
    labels = []
    inicio = 0
    for i, limite in enumerate(faixas):
        if i == 0:
            labels.append(f"0–{limite}")
        else:
            labels.append(f"{faixas[i-1]+1:,}–{limite:,}".replace(",", "."))
        inicio = limite + 1
    labels.append(f"{faixas[-1]+1:,}+".replace(",", "."))

    classe = pd.Series(labels[0], index=s.index)
    inicio = 0
    for i, limite in enumerate(faixas):
        mask = (s >= inicio) & (s <= limite)
        classe.loc[mask] = labels[i]
        inicio = limite + 1
    classe.loc[s > faixas[-1]] = labels[-1]

    labels_usados = [lab for lab in labels if (classe == lab).any()]
    return classe, labels_usados

def _norm_nome(s):
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s

# ── 1. Carrega base de mortalidade ──────────────────────────────────────────
if "f_mort" in globals() and isinstance(f_mort, pd.DataFrame):
    base_mort = f_mort.copy()
elif "CSV_MORTALIDADE" in globals():
    raw_mort = pd.read_csv(CSV_MORTALIDADE)
    base_mort = aplicar_filtros(raw_mort) if "aplicar_filtros" in globals() else raw_mort
else:
    csv_mort_padrao = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"
    raw_mort = pd.read_csv(csv_mort_padrao)
    base_mort = aplicar_filtros(raw_mort) if "aplicar_filtros" in globals() else raw_mort

if "_drop_index_cols" in globals():
    base_mort = _drop_index_cols(base_mort)

# Identifica coluna de regional e código
col_reg = _first_col(base_mort, {"Regional de Saúde","Regional de Saude","Regional","regional_de_saude","regional_saude","regional"})
col_cod = _first_col(base_mort, {"Código","Codigo","codigo_municipio","cod_municipio","codigo","cod","codigo_ibge"})

print(f"Coluna regional: {col_reg}")
print(f"Coluna código:   {col_cod}")
print("Regionais únicas na base:", sorted(base_mort[col_reg].dropna().unique()))

# ── 2. Acumula mortes por regional ──────────────────────────────────────────
anos = list(range(ANO_INI_MAPA, ANO_FIM_MAPA + 1))
cols_total = [f"Total_{a}" for a in anos if f"Total_{a}" in base_mort.columns]
if cols_total:
    acumulado_linha = base_mort[cols_total].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
else:
    cols_faixa = [c for c in base_mort.columns
                  if re.match(r"^\d{4}_.+", str(c).strip()) and int(str(c).split("_",1)[0]) in anos]
    acumulado_linha = base_mort[cols_faixa].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

base_acum = pd.DataFrame({
    "regional": base_mort[col_reg].astype(str).str.strip(),
    "mortes_1979_2024": acumulado_linha,
})

por_regional = base_acum.groupby("regional", as_index=False)["mortes_1979_2024"].sum()
por_regional["nome_norm"] = por_regional["regional"].apply(_norm_nome)

print(f"\n✅ {len(por_regional)} regionais na base de mortalidade:")
print(por_regional.sort_values("mortes_1979_2024", ascending=False).to_string(index=False))

# ── 3. Carrega malha geográfica das regionais de MG ─────────────────────────
regionais_geo = geobr.read_health_region(year=2013).to_crs(4326)
regionais_geo = regionais_geo[regionais_geo["abbrev_state"] == "MG"].copy()
regionais_geo["nome_norm"] = regionais_geo["name_health_region"].apply(_norm_nome)

# Diagnóstico de nomes sem match
nomes_geo = set(regionais_geo["nome_norm"])
nomes_csv = set(por_regional["nome_norm"])
sem_match = nomes_csv - nomes_geo
if sem_match:
    print(f"\n⚠️  Regionais da mortalidade sem match no geobr:\n  {sorted(sem_match)}")
    print(f"  Nomes disponíveis no geobr:\n  {sorted(nomes_geo)}")

# ── 4. Merge e plota ─────────────────────────────────────────────────────────
mapa_reg = regionais_geo.merge(por_regional, on="nome_norm", how="left")
mapa_reg["mortes_1979_2024"] = pd.to_numeric(mapa_reg["mortes_1979_2024"], errors="coerce").fillna(0)
mapa_reg["regional_label"]   = mapa_reg["regional"].fillna(mapa_reg["name_health_region"])

classe_reg, ordem_reg = _rotular_faixas_fixas(mapa_reg["mortes_1979_2024"], FAIXAS_REGIONAIS)
mapa_reg["classe_mortes"] = classe_reg

cores_reg = {}
labels_ordenados = list(reversed(ordem_reg))  # maior → cor mais escura
for i, lab in enumerate(labels_ordenados):
    cores_reg[lab] = PALETA_MAPA[min(i, len(PALETA_MAPA)-1)]

mapa_reg["reg_id"] = mapa_reg.index.astype(str)
geojson_reg = json.loads(mapa_reg[["reg_id","geometry"]].to_json())

fig_reg = px.choropleth(
    mapa_reg,
    geojson=geojson_reg,
    locations="reg_id",
    featureidkey="properties.reg_id",
    color="classe_mortes",
    color_discrete_map=cores_reg,
    category_orders={"classe_mortes": ordem_reg},
    hover_name="regional_label",
    hover_data={"reg_id": False, "mortes_1979_2024": ":,.0f", "classe_mortes": True},
    labels={"classe_mortes": "Mortalidade (1979–2024)", "mortes_1979_2024": "Total de mortes"},
    title="Mortalidade acumulada por Regional de Saúde (1979–2024)",
)
fig_reg.update_geos(fitbounds="locations", visible=False,
                    showcountries=False, showcoastlines=False,
                    showland=False, showframe=False)
fig_reg.update_traces(marker_line_color="#2d3436", marker_line_width=1.2)
fig_reg.update_layout(
    template="plotly_white",
    width=1250, height=820,
    margin=dict(l=10, r=10, t=80, b=10),
    title=dict(x=0.5, xanchor="center", font=dict(size=20)),
    legend=dict(orientation="h", yanchor="top", y=0.99,
                xanchor="center", x=0.5, title=dict(text="")),
)
fig_reg.show()
print("✅ Mapa por regional gerado")